# Interval propagation 3DOF reentry simulator

This notebook runs an interval propagation variant of a 3DOF spherical reentry model.

Key ideas
- Each state is an interval [lo, hi]
- Dynamics are propagated with interval Euler integration
- Multiple runs are supported by sampling different initial uncertainty boxes
- Plots show lower and upper bounds for each run and an overall envelope

Notes
- Only ASCII characters are used in text and labels
- Each plot is placed in its own notebook cell
- Code is organized into small functions for easy modification


In [12]:
# Imports and local modules
import os
import sys
import math
import random

import numpy as np
import matplotlib.pyplot as plt

# Make sure the uploaded modules are importable
MODULE_DIR = "/mnt/data"
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import constants
import AtmosphereModel
from interval_math import Interval, promote, interval_euler_step, dynamic_pressure
from math_3d import f_interval


## Configuration

Change values here to adjust:
- number of runs
- integration time step
- maximum simulation time
- initial condition uncertainty
- vehicle parameters


In [13]:
# Reproducibility for the run ensemble
RNG_SEED = 7
random.seed(RNG_SEED)

# Ensemble size
NUM_RUNS = 10

# Integrator settings
DT = 0.25            # seconds
T_MAX = 2500.0       # seconds
MAX_STEPS = int(T_MAX / DT)

# Nominal entry conditions (can be changed)
NOMINAL = {
    "alt_m": 60000.0,
    "phi_rad": 0.30,
    "lam_rad": 1.00,
    "V_ms": 7700.0,
    "gamma_rad": math.radians(-5.0),
    "chi_rad": math.radians(90.0),
}

# Uncertainty settings for sampling different runs
# Each run draws a random center offset and random halfwidth for each variable
UNCERTAINTY = {
    "alt_center_offset_m": 2000.0,   # +/- meters
    "alt_halfwidth_m": 1000.0,       # up to this halfwidth

    "V_center_offset_ms": 150.0,     # +/- m/s
    "V_halfwidth_ms": 80.0,          # up to this halfwidth

    "gamma_center_offset_deg": 0.6,  # +/- deg
    "gamma_halfwidth_deg": 0.3,      # up to this halfwidth

    "chi_center_offset_deg": 2.0,    # +/- deg
    "chi_halfwidth_deg": 1.0,        # up to this halfwidth

    "phi_center_offset_deg": 0.2,    # +/- deg
    "phi_halfwidth_deg": 0.1,        # up to this halfwidth

    "lam_center_offset_deg": 0.2,    # +/- deg
    "lam_halfwidth_deg": 0.1,        # up to this halfwidth
}

# Vehicle parameters (kept as exact scalars)
VEHICLE = {
    "mass_kg": 5000.0,
    "ref_area_m2": 10.0,
    "CD": 1.0,
    "CL": 0.30,
    "nose_radius_m": 1.0,
    "heatshield_radius_m": 2.0,
    "heatshield_rings": 6,
}

# Bank angle interval (constant for simplicity)
BANK = {
    "sigma_center_deg": 0.0,
    "sigma_halfwidth_deg": 5.0,
}


## Interval utilities

This section contains small helper functions:
- interval constructors for sampling initial boxes
- atmosphere enclosure using a layer hull
- common derived quantities


In [14]:
def make_interval(center: float, halfwidth: float) -> Interval:
    lo = float(center - halfwidth)
    hi = float(center + halfwidth)
    if lo > hi:
        lo, hi = hi, lo
    return Interval(lo, hi)

def hull_intervals(items):
    items = list(items)
    if not items:
        raise ValueError("hull_intervals: empty list")
    out = items[0]
    for iv in items[1:]:
        out = out.hull(iv)
    return out

def atmosphere_hull_from_radius(r_iv: Interval) -> dict:
    # Convert radius interval to geometric altitude interval
    z_iv = constants.intv_geometric_altitude(r_iv)

    # US Standard Atmosphere interval model returns a dict by layer index
    layers = AtmosphereModel.intv_US_Standard_ATM(z_iv)

    # Hull across layers for each quantity
    T_list = []
    p_list = []
    rho_list = []
    for k in layers.keys():
        T_list.append(layers[k]["T_K"])
        p_list.append(layers[k]["p_Pa"])
        rho_list.append(layers[k]["rho_kgm3"])

    return {
        "T_K": hull_intervals(T_list),
        "p_Pa": hull_intervals(p_list),
        "rho_kgm3": hull_intervals(rho_list),
        "z_iv": z_iv,
    }

def altitude_from_radius_iv(r_iv: Interval) -> Interval:
    return constants.intv_geometric_altitude(r_iv)

def gravity_iv(r_iv: Interval) -> Interval:
    return constants.intv_gravity(r_iv)

def tan_iv(x_iv: Interval) -> Interval:
    # Conservative tan enclosure using sin/cos with a safety check
    c = x_iv.cos()
    if c.lo <= 0.0 <= c.hi:
        # If cos crosses 0, tan can blow up.
        # Use a very wide enclosure instead of raising.
        return Interval(-1e6, 1e6)
    return x_iv.sin() / c


## Interval dynamics model

In [15]:
def aero_forces_interval(r_iv: Interval, V_iv: Interval, gamma_iv: Interval, chi_iv: Interval, sigma_iv: Interval, vehicle: dict) -> dict:
    atm = atmosphere_hull_from_radius(r_iv)
    rho = atm["rho_kgm3"]
    q = dynamic_pressure(rho, V_iv)

    S = float(vehicle["ref_area_m2"])
    CD = float(vehicle["CD"])
    CL = float(vehicle["CL"])

    D = q * (S * CD)
    L = q * (S * CL)

    # Velocity unit vector components in spherical frame
    vr = gamma_iv.sin()
    vtheta = gamma_iv.cos() * chi_iv.cos()
    vphi = gamma_iv.cos() * chi_iv.sin()

    # Drag components (opposite velocity)
    Dr = -D * vr
    Dtheta = -D * vtheta
    Dphi = -D * vphi

    # Lift basis vectors, perpendicular to velocity
    e1_r = gamma_iv.cos()
    e1_theta = -gamma_iv.sin() * chi_iv.cos()
    e1_phi = -gamma_iv.sin() * chi_iv.sin()

    e2_r = promote(0.0)
    e2_theta = chi_iv.sin()
    e2_phi = -chi_iv.cos()

    cos_sig = sigma_iv.cos()
    sin_sig = sigma_iv.sin()

    Lr = L * (e1_r * cos_sig + e2_r * sin_sig)
    Ltheta = L * (e1_theta * cos_sig + e2_theta * sin_sig)
    Lphi = L * (e1_phi * cos_sig + e2_phi * sin_sig)

    return {
        "rho": rho,
        "q": q,
        "Dr": Dr,
        "Dtheta": Dtheta,
        "Dphi": Dphi,
        "Lr": Lr,
        "Ltheta": Ltheta,
        "Lphi": Lphi,
        "atm": atm,
    }

def eom_interval(t: float, X: list, vehicle: dict, sigma_iv: Interval) -> list:
    r_iv, phi_iv, lam_iv, V_iv, gamma_iv, chi_iv = X
    m = float(vehicle["mass_kg"])

    g = gravity_iv(r_iv)
    aero = aero_forces_interval(r_iv, V_iv, gamma_iv, chi_iv, sigma_iv, vehicle)

    Dr = aero["Dr"]
    Lr = aero["Lr"]
    Ltheta = aero["Ltheta"]
    Lphi = aero["Lphi"]

    # Kinematics
    r_dot = V_iv * gamma_iv.sin()
    phi_dot = (V_iv * gamma_iv.cos() * chi_iv.sin()) / r_iv

    # Longitude uses 1 / cos(phi), which can blow up near poles.
    cphi = phi_iv.cos()
    if cphi.lo <= 0.0 <= cphi.hi:
        lam_dot = Interval(-1e-6, 1e-6)
    else:
        lam_dot = (V_iv * gamma_iv.cos() * chi_iv.cos()) / (r_iv * cphi)

    # Dynamics
    V_dot = (Dr + Lr) / m - g * gamma_iv.sin()

    # Guard division by V if V crosses zero
    if V_iv.lo <= 0.0 <= V_iv.hi:
        gamma_dot = Interval(-1e-6, 1e-6)
        chi_dot = Interval(-1e-6, 1e-6)
    else:
        gamma_dot = (Ltheta / (m * V_iv)) + (V_iv / r_iv - g / V_iv) * gamma_iv.cos()
        chi_dot = (Lphi / (m * V_iv * gamma_iv.cos())) + (V_iv / r_iv) * chi_iv.sin() * tan_iv(phi_iv)

    return [r_dot, phi_dot, lam_dot, V_dot, gamma_dot, chi_dot]


## Stop condition and simulation runner

In [16]:
def stop_condition(X):
    r, phi, lam, V, gamma, chi = X

    # geometric altitude interval
    h = constants.intv_geometric_altitude(r)

    # Stop ONLY if we are guaranteed underground
    if h.hi <= 0.0:
        return True

    # Stop ONLY if velocity is guaranteed too small
    if V.hi <= 10.0:
        return True

    return False

def sample_initial_box(run_index: int, nominal: dict, uncertainty: dict) -> tuple[list, Interval]:
    alt_center = nominal["alt_m"] + random.uniform(-uncertainty["alt_center_offset_m"], uncertainty["alt_center_offset_m"])
    V_center = nominal["V_ms"] + random.uniform(-uncertainty["V_center_offset_ms"], uncertainty["V_center_offset_ms"])

    gamma_center = nominal["gamma_rad"] + math.radians(random.uniform(-uncertainty["gamma_center_offset_deg"], uncertainty["gamma_center_offset_deg"]))
    chi_center = nominal["chi_rad"] + math.radians(random.uniform(-uncertainty["chi_center_offset_deg"], uncertainty["chi_center_offset_deg"]))
    phi_center = nominal["phi_rad"] + math.radians(random.uniform(-uncertainty["phi_center_offset_deg"], uncertainty["phi_center_offset_deg"]))
    lam_center = nominal["lam_rad"] + math.radians(random.uniform(-uncertainty["lam_center_offset_deg"], uncertainty["lam_center_offset_deg"]))

    alt_hw = random.uniform(0.25 * uncertainty["alt_halfwidth_m"], uncertainty["alt_halfwidth_m"])
    V_hw = random.uniform(0.25 * uncertainty["V_halfwidth_ms"], uncertainty["V_halfwidth_ms"])

    gamma_hw = math.radians(random.uniform(0.25 * uncertainty["gamma_halfwidth_deg"], uncertainty["gamma_halfwidth_deg"]))
    chi_hw = math.radians(random.uniform(0.25 * uncertainty["chi_halfwidth_deg"], uncertainty["chi_halfwidth_deg"]))
    phi_hw = math.radians(random.uniform(0.25 * uncertainty["phi_halfwidth_deg"], uncertainty["phi_halfwidth_deg"]))
    lam_hw = math.radians(random.uniform(0.25 * uncertainty["lam_halfwidth_deg"], uncertainty["lam_halfwidth_deg"]))

    r_iv = make_interval(constants.RADIUS_EARTH + alt_center, alt_hw)
    phi_iv = make_interval(phi_center, phi_hw)
    lam_iv = make_interval(lam_center, lam_hw)
    V_iv = make_interval(V_center, V_hw)
    gamma_iv = make_interval(gamma_center, gamma_hw)
    chi_iv = make_interval(chi_center, chi_hw)

    X0 = [r_iv, phi_iv, lam_iv, V_iv, gamma_iv, chi_iv]

    sigma_center = math.radians(BANK["sigma_center_deg"])
    sigma_hw = math.radians(BANK["sigma_halfwidth_deg"])
    sigma_iv = make_interval(sigma_center, sigma_hw)

    return X0, sigma_iv

def run_interval_simulation(run_index: int, dt: float, max_steps: int, vehicle: dict, nominal: dict, uncertainty: dict) -> dict:
    X, sigma_iv = sample_initial_box(run_index, nominal, uncertainty)

    hs = constants.HeatShield(
        radius_m=float(vehicle["heatshield_radius_m"]),
        nose_radius_m=float(vehicle["nose_radius_m"]),
        num_rings=int(vehicle["heatshield_rings"]),
        radial_exp=1.0
    )

    t_hist = []
    X_hist = []
    derived = {
        "alt_m": [],
        "rho": [],
        "q": [],
        "g": [],
        "qdot_max": [],
        "Q_max": [],
        "qdot_mean": [],
        "width_r": [],
        "width_V": [],
        "width_alt": [],
        "width_gamma": [],
        "width_chi": [],
    }

    t = 0.0
    for step in range(max_steps):
        t_hist.append(t)
        X_hist.append(X)

        r_iv, phi_iv, lam_iv, V_iv, gamma_iv, chi_iv = X

        alt_iv = altitude_from_radius_iv(r_iv)
        atm = atmosphere_hull_from_radius(r_iv)
        rho_iv = atm["rho_kgm3"]
        q_iv = dynamic_pressure(rho_iv, V_iv)
        g_iv = gravity_iv(r_iv)

        hs.update(rho_iv, V_iv, dt)

        derived["alt_m"].append(alt_iv)
        derived["rho"].append(rho_iv)
        derived["q"].append(q_iv)
        derived["g"].append(g_iv)
        derived["qdot_max"].append(hs.qdot_max())
        derived["Q_max"].append(hs.Q_max())
        derived["qdot_mean"].append(hs.qdot_mean())

        derived["width_r"].append(r_iv.width())
        derived["width_V"].append(V_iv.width())
        derived["width_alt"].append(alt_iv.width())
        derived["width_gamma"].append(gamma_iv.width())
        derived["width_chi"].append(chi_iv.width())

        if stop_condition(X):
            break

        def f_local(X_local):
            return eom_interval(t, X_local, vehicle, sigma_iv)

        X = interval_euler_step(X, dt, f_local)
        t += dt

    return {
        "run_index": run_index,
        "t": np.array(t_hist, dtype=float),
        "X": X_hist,
        "derived": derived,
        "sigma_iv": sigma_iv,
    }


## Run the ensemble

In [17]:
runs = []

for i in range(NUM_RUNS):
    try:
        out = run_interval_simulation(
            run_index=i,
            dt=DT,
            max_steps=MAX_STEPS,
            vehicle=VEHICLE,
            nominal=NOMINAL,
            uncertainty=UNCERTAINTY,
        )
        runs.append(out)

    except ValueError as e:
        # Interval math domain failure is expected when altitude interval
        # crosses invalid physical regions (e.g. log of non-positive values).
        # Terminate this run cleanly and keep the ensemble alive.
        print(f"Run {i} terminated early due to domain violation: {e}")

# Quick check (only if at least one run succeeded)
if len(runs) > 0:
    len(runs), runs[0]["t"][-1]
else:
    len(runs), None


Run 0 terminated early due to domain violation: log undefined for interval containing non-positive values: Interval(-0.05768789208591964, 0.9394895797892183)
Run 1 terminated early due to domain violation: log undefined for interval containing non-positive values: Interval(-0.17040324034635457, 1.1003049890044234)
Run 2 terminated early due to domain violation: log undefined for interval containing non-positive values: Interval(-0.27601784132413565, 1.2487551819910077)
Run 3 terminated early due to domain violation: log undefined for interval containing non-positive values: Interval(-0.19835908193759766, 1.1491362494121142)
Run 4 terminated early due to domain violation: log undefined for interval containing non-positive values: Interval(-0.34770380374203275, 1.194620582477743)
Run 5 terminated early due to domain violation: log undefined for interval containing non-positive values: Interval(-0.22347772836327995, 1.0727490070242593)
Run 6 terminated early due to domain violation: log u

## Plot helpers

In [18]:
def extract_series(runs, selector_fn):
    series = []
    for run in runs:
        t = run["t"]
        lo = np.zeros_like(t)
        hi = np.zeros_like(t)
        for j in range(len(t)):
            iv = selector_fn(run, j)
            lo[j] = iv.lo
            hi[j] = iv.hi
        series.append({"t": t, "lo": lo, "hi": hi})
    return series

def common_time_grid(series, n=1200):
    t_end = min(s["t"][-1] for s in series)
    return np.linspace(0.0, t_end, n)

def interp_to_grid(s, t_grid):
    lo_i = np.interp(t_grid, s["t"], s["lo"])
    hi_i = np.interp(t_grid, s["t"], s["hi"])
    return lo_i, hi_i

def plot_interval_ensemble(series, title, ylabel, xlabel="time s"):
    t_grid = common_time_grid(series, n=1200)
    lo_stack = []
    hi_stack = []

    plt.figure(figsize=(10, 5))

    for s in series:
        lo_i, hi_i = interp_to_grid(s, t_grid)
        lo_stack.append(lo_i)
        hi_stack.append(hi_i)
        plt.plot(t_grid, lo_i, linewidth=0.9, alpha=0.35)
        plt.plot(t_grid, hi_i, linewidth=0.9, alpha=0.35)

    lo_stack = np.vstack(lo_stack)
    hi_stack = np.vstack(hi_stack)

    env_lo = np.min(lo_stack, axis=0)
    env_hi = np.max(hi_stack, axis=0)

    plt.fill_between(t_grid, env_lo, env_hi, alpha=0.15)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.show()


## End state summary

This cell prints the final interval state for each run.


In [19]:
def fmt_iv(iv: Interval) -> str:
    return f"[{iv.lo:.6g}, {iv.hi:.6g}]"

for run in runs:
    Xf = run["X"][-1]
    altf = run["derived"]["alt_m"][-1]
    Vf = Xf[3]
    gf = Xf[4]
    print("run", run["run_index"], "t_end_s", f"{run['t'][-1]:.2f}")
    print("  altitude_m", fmt_iv(altf))
    print("  speed_ms", fmt_iv(Vf))
    print("  gamma_rad", fmt_iv(gf))
    print("  heading_rad", fmt_iv(Xf[5]))
    print("")


In [20]:
# ---------------------------------------------
# Robust extraction of interval radius series
# ---------------------------------------------

def safe_selector(run, j):
    """
    Safely extract radius interval at step j.
    Returns None if unavailable.
    """
    try:
        return run["X"][j][0]  # radius interval
    except (KeyError, IndexError, TypeError):
        return None


## interval propagation


In [21]:
# ===== GLOBAL SETUP (RUN FIRST) =====

import math
from interval_math import Interval
import constants

dt = 0.05
max_steps = 2000
num_runs = 1

X0 = [
    Interval(constants.RADIUS_EARTH + 58_000.0,
             constants.RADIUS_EARTH + 62_000.0),
    Interval(0.30, 0.31),
    Interval(1.00, 1.01),
    Interval(7600.0, 7800.0),
    Interval(math.radians(-5.2), math.radians(-4.8)),
    Interval(math.radians(85.0), math.radians(95.0)),
]


In [ ]:
# ---------------- Interval propagation ----------------

series = []

num_runs = 1

for run_idx in range(num_runs):
    X = X0[:]  # copy initial interval state
    t = 0.0

    t_hist = []
    r_hist = []

    alive = True

    for step in range(max_steps):
        if stop_condition(X):
            break

        try:
            X = interval_euler_step(
                X,
                dt,
                lambda X_: f_interval(t, X_, VEHICLE, BANK)
            )
        except ValueError:
            # Interval division failure → terminate run safely
            alive = False
            break

        r = X[0]
        if not isinstance(r, Interval):
            alive = False
            break

        t_hist.append(t)
        r_hist.append(r)

        t += dt

    # Only keep trajectories that actually produced data
    if alive and len(r_hist) > 1:
        series.append({
            "t": t_hist,
            "intervals": r_hist,
            "label": f"run {run_idx}"
        })

# ---------------- Plot ----------------

if len(series) == 0:
    raise RuntimeError(
        "No plottable interval trajectories found.\n"
        "All runs terminated due to interval blow-up (division by zero or cos() crossing)."
    )

plot_interval_ensemble(
    series,
    title="Radius interval bounds vs time",
    ylabel="radius (m)"
)


NameError: name 'vehicle_params' is not defined

In [ ]:

selector = lambda run, j: run["derived"]["alt_m"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Altitude interval bounds vs time", ylabel="altitude m")


In [ ]:

selector = lambda run, j: run["X"][j][3]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Speed interval bounds vs time", ylabel="speed m s")


In [ ]:

selector = lambda run, j: run["X"][j][4]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Flight path angle interval bounds vs time", ylabel="flight path angle rad")


In [ ]:

selector = lambda run, j: run["X"][j][5]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Heading angle interval bounds vs time", ylabel="heading angle rad")


In [ ]:

selector = lambda run, j: run["X"][j][1]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Latitude interval bounds vs time", ylabel="latitude rad")


In [ ]:

selector = lambda run, j: run["X"][j][2]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Longitude interval bounds vs time", ylabel="longitude rad")


In [ ]:

selector = lambda run, j: run["derived"]["rho"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Density interval bounds vs time", ylabel="density kg m3")


In [ ]:

selector = lambda run, j: run["derived"]["q"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Dynamic pressure interval bounds vs time", ylabel="dynamic pressure Pa")


In [ ]:

selector = lambda run, j: run["derived"]["g"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Gravity interval bounds vs time", ylabel="gravity m s2")


In [ ]:

selector = lambda run, j: run["derived"]["qdot_max"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Heat rate max interval bounds vs time", ylabel="heat rate W m2")


In [ ]:

selector = lambda run, j: run["derived"]["Q_max"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Heat load max interval bounds vs time", ylabel="heat load J m2")


In [ ]:

selector = lambda run, j: run["derived"]["qdot_mean"][j]
series = extract_series(runs, selector)
plot_interval_ensemble(series, title="Heat rate mean interval bounds vs time", ylabel="heat rate mean W m2")


In [ ]:

# Altitude interval width vs time
plt.figure(figsize=(10, 5))
t_end = min(run["t"][-1] for run in runs)
t_grid = np.linspace(0.0, t_end, 1200)

stack = []
for run in runs:
    t = run["t"]
    w = np.array(run["derived"]["width_alt"], dtype=float)
    w_i = np.interp(t_grid, t, w)
    stack.append(w_i)
    plt.plot(t_grid, w_i, linewidth=0.9, alpha=0.35)

stack = np.vstack(stack)
env_lo = np.min(stack, axis=0)
env_hi = np.max(stack, axis=0)
plt.fill_between(t_grid, env_lo, env_hi, alpha=0.15)

plt.title("Altitude interval width vs time")
plt.xlabel("time s")
plt.ylabel("width m")
plt.grid(True)
plt.show()


In [ ]:

# Speed interval width vs time
plt.figure(figsize=(10, 5))
t_end = min(run["t"][-1] for run in runs)
t_grid = np.linspace(0.0, t_end, 1200)

stack = []
for run in runs:
    t = run["t"]
    w = np.array(run["derived"]["width_V"], dtype=float)
    w_i = np.interp(t_grid, t, w)
    stack.append(w_i)
    plt.plot(t_grid, w_i, linewidth=0.9, alpha=0.35)

stack = np.vstack(stack)
env_lo = np.min(stack, axis=0)
env_hi = np.max(stack, axis=0)
plt.fill_between(t_grid, env_lo, env_hi, alpha=0.15)

plt.title("Speed interval width vs time")
plt.xlabel("time s")
plt.ylabel("width m s")
plt.grid(True)
plt.show()


In [ ]:

# Flight path angle interval width vs time
plt.figure(figsize=(10, 5))
t_end = min(run["t"][-1] for run in runs)
t_grid = np.linspace(0.0, t_end, 1200)

stack = []
for run in runs:
    t = run["t"]
    w = np.array(run["derived"]["width_gamma"], dtype=float)
    w_i = np.interp(t_grid, t, w)
    stack.append(w_i)
    plt.plot(t_grid, w_i, linewidth=0.9, alpha=0.35)

stack = np.vstack(stack)
env_lo = np.min(stack, axis=0)
env_hi = np.max(stack, axis=0)
plt.fill_between(t_grid, env_lo, env_hi, alpha=0.15)

plt.title("Flight path angle interval width vs time")
plt.xlabel("time s")
plt.ylabel("width rad")
plt.grid(True)
plt.show()
